In [ ]:
import duckdb
from pathlib import Path

# Folder containing this script
ROOT = Path(__file__).resolve().parent

set_a = ROOT / "training_setA"/"training"
set_b = ROOT / "training_setB"/"training_setB"

output = ROOT / "physionet_deterioration.parquet"

print("Root:", ROOT)
print("Set A:", set_a)
print("Set B:", set_b)
print("Output:", output)

# Validate folders
if not set_a.exists():
    raise FileNotFoundError(f"training_setA not found: {set_a}")

if not set_b.exists():
    raise FileNotFoundError(f"training_setB not found: {set_b}")

con = duckdb.connect()

query = f"""
COPY (
    SELECT
        * EXCLUDE (filename),

        CASE
            WHEN filename LIKE '%training_setA%' THEN 'A'
            WHEN filename LIKE '%training_setB%' THEN 'B'
            ELSE 'UNKNOWN'
        END AS source_set,

        CASE
            WHEN filename LIKE '%training_setA%' THEN
                'A_' || regexp_extract(filename, 'p[0-9]+', 0)

            WHEN filename LIKE '%training_setB%' THEN
                'B_' || regexp_extract(filename, 'p[0-9]+', 0)

            ELSE
                regexp_extract(filename, 'p[0-9]+', 0)
        END AS patient_id

    FROM read_csv(
        [
            '{set_a.as_posix()}/*.psv',
            '{set_b.as_posix()}/*.psv'
        ],
        delim='|',
        header=true,
        union_by_name=true,
        filename=true,
        nullstr='NaN'
    )
)
TO '{output.as_posix()}'
(
    FORMAT PARQUET,
    COMPRESSION ZSTD
);
"""

print("\nCombining patient files...")

con.execute(query)

print("\nSUCCESS")
print("Created:", output)